# OutBoxML: модель 2 (TARGET_FREQ / TARGET_SEV)

Конфиги генерируются из артефактов train_loop_new (фичи + HPO). **Без калибровки** — в прод идут сырые `predict_proba` / `predict`.

**Прод-ансамбль** = refit на train ∪ 85% более старого test; holdout для финэффекта = **15% самых свежих** дат test.

После prod-fit: **FactorsPlot** (`ResultExport.plots`, `plot_type=1`), **EMailDSResult**, parquet для сервиса = `df` + `preds_cf` / `preds_rg`.

Parity-fit — только для сверки финэффекта на полном holdout Test (как collect C3).

На сервисе: перед prepare_dataset — `apply_frozen_dq_bounds` из `querulus_dq_bounds_{version}.json`.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
OUTBOXML_ROOT = PROJECT_ROOT.parent.parent
for _p in (SRC, OUTBOXML_ROOT, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
print("PROJECT_ROOT", PROJECT_ROOT)
print("OUTBOXML_ROOT", OUTBOXML_ROOT)


In [ ]:
import json
import pickle
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format

from outboxml.core.email import EMailDSResult
from outboxml.core.prepared_datasets import PrepareDataset
from outboxml.data_subsets import DataPreprocessor
from outboxml.datasets_manager import DataSetsManager
from outboxml.export_results import ResultExport

from querulus.training.build_outboxml_configs import (
    dataframe_for_dsm,
    default_model_version,
    prepare_datasets_from_config,
    ensure_predictable_model,
    unwrap_estimator,
    write_outboxml_configs,
)
from querulus.training.outboxml_metrics import display_dsm_collect_metrics
from querulus.features.data_quality import write_service_dq_bounds
from querulus.fin_effect import (
    create_summary_table,
    export_business_html,
    print_best_threshold_report,
    resolve_fin_effect_config,
    run_fin_effect_pipeline,
)


In [ ]:
MODEL_VERSION = default_model_version(business="2", increment="v1")
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"
RESULTS_DIR = PROJECT_ROOT / "integration" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_VERSION", MODEL_VERSION)
print("DATASET_PATH", DATASET_PATH)

In [ ]:
df = dataframe_for_dsm(pd.read_parquet(DATASET_PATH))
print("df", df.shape)
built = write_outboxml_configs(
    df,
    version=MODEL_VERSION,
    parquet_path=str(DATASET_PATH.as_posix()),
)
periods = built["periods"]
CF_NAME = built["cf_name"]
RG_NAME = built["rg_name"]
display(Markdown("### Периоды (даты / n / n_positive по TARGET_FREQ)"))
display(periods["table"])
print("cutoff prod_cal", periods["prod_cutoff"])
print("cf", built["cf_path"])
print("rg", built["rg_path"])
print("cf_prod", built["cf_prod_path"])
print("rg_prod", built["rg_prod_path"])

In [ ]:
def _patch_dsm_models(dsm):
    for name, res in dsm.get_result().items():
        res.model = ensure_predictable_model(res.model)


def _prepared_X(dsm, model_name, data, *, ignore_row_filter=False):
    """Обёртка: признаки DSM. Для severity на полном Test — ignore_row_filter=True."""
    from querulus.training.outboxml_metrics import prepare_dsm_features

    return prepare_dsm_features(
        dsm, model_name, data, ignore_row_filter=ignore_row_filter
    )


def _predict_cf(dsm, model_name, data):
    from querulus.training.outboxml_metrics import predict_dsm_series

    return predict_dsm_series(
        dsm,
        model_name,
        data,
        task_type="classification",
        ignore_row_filter=False,
    )


def _predict_rg(dsm, model_name, data):
    """Severity на всех строках data (без фильтра TARGET_SEV > 0 из обучения)."""
    from querulus.training.outboxml_metrics import predict_dsm_series

    return predict_dsm_series(
        dsm,
        model_name,
        data,
        task_type="regression",
        ignore_row_filter=True,
    )


def _fin_effect_table(df_all, index, proba, sev, *, threshold=None, title=""):
    """Финэффект: proba и sev на одном index (полный holdout), сводка = агрегация frame."""
    cfg = resolve_fin_effect_config(
        df_all,
        frequency_target="TARGET_FREQ",
        severity_target="TARGET_SEV",
    )
    common = (
        pd.Index(index)
        .intersection(proba.dropna().index)
        .intersection(sev.dropna().index)
        .intersection(df_all.index)
    )
    if len(common) == 0:
        raise ValueError(
            "Нет пересечения index с proba/sev: проверь index предсказаний."
        )
    coverage = len(common) / max(len(pd.Index(index)), 1)
    if coverage < 0.95:
        raise ValueError(
            f"pred покрывает только {len(common)}/{len(index)} строк ({coverage:.1%}). "
            "Для severity нужен predict без data_filter_condition "
            "(ignore_row_filter=True в _predict_rg)."
        )
    if len(common) < len(index):
        print(
            f"[fin_effect] строк с pred: {len(common)}/{len(index)} "
            f"(отброшено без proba/sev: {len(index) - len(common)})"
        )
    aligned = df_all.loc[common]
    fe = run_fin_effect_pipeline(
        aligned,
        proba.reindex(common),
        sev.reindex(common),
        aligned["TARGET_FREQ"],
        threshold=threshold,
        config=cfg,
    )
    if title:
        display(Markdown(f"### {title}"))
    print_best_threshold_report(fe)
    summary = fe.summary_table(cfg)
    display(summary.style.format("{:,.0f}", subset=summary.columns[3:], na_rep="—"))
    print(
        f"проверка Σ: model={summary['ФИН. ЭФФЕКТ МОДЕЛЬ'].sum():,.0f} "
        f"(отчёт {fe.model_effect_total:,.0f}), "
        f"fact={summary['ФИН. ЭФФЕКТ ФАКТ'].sum():,.0f} "
        f"(отчёт {fe.fact_effect_total:,.0f}), "
        f"экон.={summary['Экономия'].sum():,.0f} "
        f"(отчёт net {fe.net_effect:,.0f})"
    )
    n_pos = int((aligned["TARGET_FREQ"] == 1).sum())
    n_neg = int((aligned["TARGET_FREQ"] == 0).sum())
    print(f"holdout в расчёте: n={len(aligned)} pos={n_pos} neg={n_neg}")
    return fe, summary


## Parity: DSM на train_core∪val, test = полный holdout

Эти модели **не** идут в прод-pickle. Нужны для таблицы финэффекта на том же Test, что collect C3.


In [ ]:
from configs import config as querulus_outboxml_config

dsm_cf = DataSetsManager(
    config_name=str(built["cf_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_path"]),
)
dsm_cf.load_dataset(data=df)
dsm_cf.fit_models()
_patch_dsm_models(dsm_cf)

dsm_rg = DataSetsManager(
    config_name=str(built["rg_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_path"]),
)
dsm_rg.load_dataset(data=df)
dsm_rg.fit_models()
_patch_dsm_models(dsm_rg)
display_dsm_collect_metrics(dsm_cf, CF_NAME, task_type="classification", title=f"parity {CF_NAME}")
display_dsm_collect_metrics(dsm_rg, RG_NAME, task_type="regression", title=f"parity {RG_NAME}")
print("parity fit done", CF_NAME, RG_NAME)


In [ ]:
test_idx = periods["splits"].test
proba_test = _predict_cf(dsm_cf, CF_NAME, df.loc[test_idx])
sev_test = _predict_rg(dsm_rg, RG_NAME, df.loc[test_idx])
print("test n", len(test_idx), "proba", proba_test.shape, "sev", sev_test.shape)


## Таблицы финэффекта (сверка глазами)

1. Collect после HPO на полном Test — если в ядре есть `fin_effect_b`.
2. OutBoxML parity на сырых proba (полный Test).


In [ ]:
_fe_collect = globals().get("fin_effect_b")
if _fe_collect is not None:
    display(Markdown("### Collect HPO / блок C3 (полный Test)"))
    print_best_threshold_report(_fe_collect)
    _cfg_c = globals().get("FIN_EFFECT_CONFIG_B")
    if _cfg_c is not None:
        _sum_c = create_summary_table(_fe_collect.frame, _cfg_c)
        display(_sum_c.style.format("{:,.0f}", subset=_sum_c.columns[3:], na_rep="—"))
else:
    print("fin_effect_b нет в kernel — прогон collect C3 или смотри только таблицу OutBoxML ниже")

fe_parity, _ = _fin_effect_table(
    df,
    test_idx,
    proba_test,
    sev_test,
    title="OutBoxML parity (полный Test, сырые proba)",
)

_fe_html = _fe_collect if _fe_collect is not None else fe_parity
_cfg_html = globals().get("FIN_EFFECT_CONFIG_B")
if _cfg_html is None:
    _cfg_html = resolve_fin_effect_config(
        df, frequency_target="TARGET_FREQ", severity_target="TARGET_SEV"
    )
_html_path = export_business_html(
    _fe_html,
    _cfg_html,
    path=PROJECT_ROOT / "notebooks" / "fin_effect_detailed.html",
    subtitle=(
        "Collect, блок финансового эффекта на полном Test"
        if _fe_collect is not None
        else "OutBoxML parity, сырые proba, полный Test"
    ),
)
print(f"HTML для бизнеса: {_html_path}")


## Prod-refit (идёт в ансамбль)

Обучение: исходный train + test **до cutoff** (85% более старых строк test по дате).
Holdout финэффекта / метрик: **15% самых свежих** дат test (без калибровки).


In [ ]:
dsm_cf_prod = DataSetsManager(
    config_name=str(built["cf_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_prod_path"]),
)
dsm_cf_prod.load_dataset(data=df)
dsm_cf_prod.fit_models()
_patch_dsm_models(dsm_cf_prod)

dsm_rg_prod = DataSetsManager(
    config_name=str(built["rg_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_prod_path"]),
)
dsm_rg_prod.load_dataset(data=df)
dsm_rg_prod.fit_models()
_patch_dsm_models(dsm_rg_prod)

prod_holdout_idx = df.index[
    (pd.to_datetime(df[periods["date_column"]], errors="coerce") >= pd.Timestamp(periods["prod_test_period"][0]))
    & (pd.to_datetime(df[periods["date_column"]], errors="coerce") <= pd.Timestamp(periods["prod_test_period"][1]))
]
print("prod_holdout n", len(prod_holdout_idx))

proba_prod = _predict_cf(dsm_cf_prod, CF_NAME, df.loc[prod_holdout_idx])
sev_prod = _predict_rg(dsm_rg_prod, RG_NAME, df.loc[prod_holdout_idx])
fe_prod, _ = _fin_effect_table(
    df,
    prod_holdout_idx,
    proba_prod,
    sev_prod,
    title="Prod-refit (15% freshest holdout, сырые proba)",
)

display_dsm_collect_metrics(
    dsm_cf_prod, CF_NAME, task_type="classification", title=f"prod {CF_NAME}"
)
display_dsm_collect_metrics(
    dsm_rg_prod, RG_NAME, task_type="regression", title=f"prod {RG_NAME}"
)


## FactorsPlot + EMailDSResult (prod)

`ResultExport.plots(..., plot_type=1)` — факторные графики OutBoxML.
`EMailDSResult(...).success_mail(...)` — метрики + cohort-графики на почту из `configs/config.py`.


In [ ]:
def _plot_features(dsm, model_name, *, n_num=6, n_cat=6):
    """Фичи для FactorsPlot: из data_subset (в JSON targetslices пустые)."""
    subset = dsm.get_result()[model_name].data_subset
    nums = list(subset.features_numerical or [])[:n_num]
    cats = list(subset.features_categorical or [])[:n_cat]
    return nums + cats


# FactorsPlot (plot_type=1) для CF и RG
export_cf = ResultExport(ds_manager=dsm_cf_prod, config=querulus_outboxml_config)
export_rg = ResultExport(ds_manager=dsm_rg_prod, config=querulus_outboxml_config)
cf_plot_feats = _plot_features(dsm_cf_prod, CF_NAME)
rg_plot_feats = _plot_features(dsm_rg_prod, RG_NAME)
print("FactorsPlot features CF:", cf_plot_feats)
print("FactorsPlot features RG:", rg_plot_feats)
fig_cf_factors = export_cf.plots(
    model_name=CF_NAME,
    features=cf_plot_feats,
    plot_type=1,
    bins_for_numerical_features=5,
    use_exposure=False,
)
fig_rg_factors = export_rg.plots(
    model_name=RG_NAME,
    features=rg_plot_feats,
    plot_type=1,
    bins_for_numerical_features=5,
    use_exposure=False,
)
print("FactorsPlot CF:", type(fig_cf_factors), "RG:", type(fig_rg_factors))

# EMailDSResult — отчёт по обеим prod-моделям
_prod_results = {}
_prod_results.update(dsm_cf_prod.get_result())
_prod_results.update(dsm_rg_prod.get_result())
try:
    EMailDSResult(
        config=querulus_outboxml_config,
        ds_manager_result=_prod_results,
    ).success_mail(group_name=f"querulus_{MODEL_VERSION}")
    print("EMailDSResult: письмо отправлено")
except Exception as exc:
    print(f"[warn] EMailDSResult не отправился: {type(exc).__name__}: {exc}")


## Export pickle + df_for_service (prod)


In [ ]:
cf_export = dsm_cf_prod.get_result()[CF_NAME].dict_for_prod_export()
rg_export = dsm_rg_prod.get_result()[RG_NAME].dict_for_prod_export()
cf_export["model"] = ensure_predictable_model(cf_export["model"])
rg_export["model"] = ensure_predictable_model(rg_export["model"])

cf_pkl = RESULTS_DIR / f"querulus_cf_for_prod_{MODEL_VERSION}.pickle"
rg_pkl = RESULTS_DIR / f"querulus_rg_for_prod_{MODEL_VERSION}.pickle"
ans_pkl = RESULTS_DIR / f"querulus_ansamble_{MODEL_VERSION}.pickle"
dq_report = PROJECT_ROOT / "data" / "processed" / "data_quality_report.json"
dq_bounds_pkl = RESULTS_DIR / f"querulus_dq_bounds_{MODEL_VERSION}.json"
if dq_report.exists():
    write_service_dq_bounds(
        dq_bounds_pkl,
        model_version=MODEL_VERSION,
        report_path=dq_report,
    )
    print("dq_bounds", dq_bounds_pkl)
else:
    print("[warn] нет data_quality_report.json — dq_bounds не записан")
    dq_bounds_pkl = None

cf_pkl.write_bytes(pickle.dumps([cf_export]))
rg_pkl.write_bytes(pickle.dumps([rg_export]))
ensemble = [deepcopy(cf_export), deepcopy(rg_export)]
ans_pkl.write_bytes(pickle.dumps(ensemble))

# Датафрейм для сервиса: итоговый df + ответы prod-моделей
df_service = df.copy()
df_service["preds_cf"] = _predict_cf(dsm_cf_prod, CF_NAME, df)
df_service["preds_rg"] = _predict_rg(dsm_rg_prod, RG_NAME, df)
service_df_path = PROJECT_ROOT / "data" / "processed" / f"df_for_service_{MODEL_VERSION}.parquet"
df_service.to_parquet(service_df_path, index=True)
print(
    "df_for_service", service_df_path,
    "shape", df_service.shape,
    "preds_cf na%", float(df_service["preds_cf"].isna().mean()),
    "preds_rg na%", float(df_service["preds_rg"].isna().mean()),
)

meta = {
    "model_version": MODEL_VERSION,
    "periods": {
        k: list(v) if isinstance(v, tuple) else v
        for k, v in periods.items()
        if k in {
            "parity_train_period",
            "parity_test_period",
            "prod_train_period",
            "prod_test_period",
            "prod_cutoff",
            "date_column",
            "cal_period",
        }
    },
    "cf_name": CF_NAME,
    "rg_name": RG_NAME,
    "best_threshold": float(fe_prod.best_threshold),
    "calibration": None,
    "dq_bounds": str(dq_bounds_pkl) if dq_bounds_pkl else None,
    "service_df": str(service_df_path),
    "preds_cf_col": "preds_cf",
    "preds_rg_col": "preds_rg",
    "artifacts": {
        "cf": str(cf_pkl),
        "rg": str(rg_pkl),
        "ensemble": str(ans_pkl),
        "service_df": str(service_df_path),
    },
}
(RESULTS_DIR / f"querulus_meta_{MODEL_VERSION}.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print(
    "wrote",
    cf_pkl.name,
    rg_pkl.name,
    ans_pkl.name,
    dq_bounds_pkl.name if dq_bounds_pkl else None,
    service_df_path.name,
)
print("best_threshold=", meta["best_threshold"])


## DQ bounds для сервиса

Файл querulus_dq_bounds_{version}.json — заборы с **сборки** df_final_3 (тот же winsorize, на котором учили).

**Контракт FastAPI (позже):** сырой вектор → apply_frozen_dq_bounds → prepare_dataset / predict. IQR на заявке не считать.


## Roundtrip predict


In [ ]:
sample = df.loc[prod_holdout_idx].head(20)
res_cf = dsm_cf_prod.model_predict(sample, CF_NAME)
res_rg = dsm_rg_prod.model_predict(sample, RG_NAME)
p_cf = _predict_cf(dsm_cf_prod, CF_NAME, sample)
p_rg = _predict_rg(dsm_rg_prod, RG_NAME, sample)
# model_predict -> DSManagerResult: preds in .y_pred (not .prediction)
dsm_cf_hat = res_cf.y_pred.reindex(sample.index)
dsm_rg_hat = res_rg.y_pred.reindex(sample.index)
display(
    pd.DataFrame(
        {
            "dsm_cf": dsm_cf_hat,
            "proba_cf": p_cf.reindex(sample.index),
            "dsm_rg": dsm_rg_hat,
            "pred_sev": p_rg.reindex(sample.index),
        }
    ).head()
)
